# Floci worker workflow sample

This notebook exercises the current analysis workflow against the local Floci stack using the repository's two sample files:

- `tests/samples/dlptest-name-dob-email.pdf`
- `tests/samples/Passport_of_Austria_(2024)_data_page.jpg`

It uses real Floci S3 and DynamoDB resources, the real worker orchestration, rasterization, checkpoint, manifest, and lifecycle code, and deterministic injected OCR/PII engines. The deterministic engines keep the notebook runnable from the repository Python environment without installing the heavy worker image dependencies locally.

Start Floci before running:

```bash
make floci-up
```

The notebook creates or reuses `redact-jobs` and `redact-files`, clears only the two notebook job prefixes, runs analysis, then draws persisted redaction regions on the rasterized page artifacts.

## 1. Imports and repository setup

Run the notebook from the repository root or any subdirectory. The setup below finds `src/redact_api` and imports the same code used by tests and worker smoke commands.

In [ ]:
from __future__ import annotations

import io
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError, EndpointConnectionError
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

repo_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src" / "redact_api").is_dir()
)
sys.path.insert(0, str(repo_root / "src"))

from redact_api.aws import DynamoJobRepository, S3ObjectRepository
from redact_api.domain import Job, JobStatus, SourceType
from redact_api.worker import (
    OcrLine,
    OcrPageText,
    PageArtifact,
    PiiEntity,
    analyze_source,
    page_artifact_key,
    PaddleOcrEngine,
    Gliner2PiiDetector,
)

NOW = datetime(2026, 7, 3, 12, 0, tzinfo=UTC)
OWNER_ID = "notebook-user"
repo_root

## 2. Connect to Floci and prepare resources

These names match the local integration-test stack. The cleanup function deletes only `users/notebook-user/jobs/<job-id>/...` objects and the matching notebook job records.

In [ ]:
FLOCI_URL = "http://localhost:4566"
REGION = "eu-west-1"
TABLE_NAME = "redact-jobs"
BUCKET_NAME = "redact-files"

aws = {
    "endpoint_url": FLOCI_URL,
    "region_name": REGION,
    "aws_access_key_id": "test",
    "aws_secret_access_key": "test",
}

dynamodb = boto3.resource("dynamodb", **aws)
s3 = boto3.client(
    "s3",
    config=Config(s3={"addressing_style": "path"}),
    **aws,
)

try:
    assert s3.list_buckets()["ResponseMetadata"]["HTTPStatusCode"] == 200
    print("✅ Floci is available.")  # add checkmark emoji
except EndpointConnectionError as error:
    raise RuntimeError("Floci is unavailable. Start it with `make floci-up`.") from error

existing_tables = dynamodb.meta.client.list_tables()["TableNames"]
if TABLE_NAME not in existing_tables:
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "id", "KeyType": "HASH"}],
        AttributeDefinitions=[
            {"AttributeName": "id", "AttributeType": "S"},
            {"AttributeName": "owner_id", "AttributeType": "S"},
            {"AttributeName": "created_sort", "AttributeType": "S"},
        ],
        GlobalSecondaryIndexes=[
            {
                "IndexName": "owner-created-index",
                "KeySchema": [
                    {"AttributeName": "owner_id", "KeyType": "HASH"},
                    {"AttributeName": "created_sort", "KeyType": "RANGE"},
                ],
                "Projection": {"ProjectionType": "ALL"},
            }
        ],
        BillingMode="PAY_PER_REQUEST",
    )
    table.wait_until_exists()
else:
    table = dynamodb.Table(TABLE_NAME)

existing_buckets = {bucket["Name"] for bucket in s3.list_buckets().get("Buckets", [])}
if BUCKET_NAME not in existing_buckets:
    s3.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": REGION},
    )

jobs = DynamoJobRepository(table)
objects = S3ObjectRepository(s3, bucket=BUCKET_NAME)


def clear_notebook_job(job_id: str) -> None:
    table.delete_item(Key={"id": job_id})
    prefix = f"users/{OWNER_ID}/jobs/{job_id}/"
    response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    keys = [{"Key": item["Key"]} for item in response.get("Contents", [])]
    if keys:
        s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})

{"endpoint": FLOCI_URL, "table": TABLE_NAME, "bucket": BUCKET_NAME}

## 3. Define deterministic OCR and PII engines

The worker accepts injected engines through protocols. These engines produce stable text and PII spans while the rest of the workflow writes to Floci, rasterizes real sample pages, persists OCR JSON, creates manifests, checkpoints `completed_pages`, and marks jobs `ready`.

To test the heavy production inference path, build/run the worker image or install the worker runtime, then replace these injected engines with `PaddleOcrEngine()` and `Gliner2PiiDetector()`.

In [ ]:
# class SampleOcrEngine:
#     name = "sample-ocr"
#     model = "deterministic-notebook"

#     def recognize(self, page: PageArtifact, content: bytes) -> tuple[OcrLine, ...]:
#         width = float(page.width)
#         height = float(page.height)
#         y1 = max(10.0, height * 0.12)
#         y2 = y1 + max(16.0, height * 0.04)
#         y3 = y2 + max(18.0, height * 0.05)
#         y4 = y3 + max(16.0, height * 0.04)
#         x1 = max(10.0, width * 0.10)
#         x2 = min(width - 10.0, width * 0.90)

#         if "passport-jpeg" in page.key:
#             lines = (
#                 "Passport A12345678",
#                 "Name Erika Mustermann DOB 1988-08-12",
#             )
#         else:
#             lines = (
#                 "Jane Public DOB 2000-01-01",
#                 "Email jane.public@example.com",
#             )

#         return (
#             OcrLine(lines[0], 0.99, ((x1, y1), (x2, y1), (x2, y2), (x1, y2))),
#             OcrLine(lines[1], 0.98, ((x1, y3), (x2, y3), (x2, y4), (x1, y4))),
#         )


# class SamplePiiDetector:
#     name = "sample-pii"
#     model = "deterministic-notebook"

#     def detect(self, page: OcrPageText) -> tuple[PiiEntity, ...]:
#         targets = {
#             "full_name": ["Jane Public", "Erika Mustermann"],
#             "date_of_birth": ["2000-01-01", "1988-08-12"],
#             "email": ["jane.public@example.com"],
#             "passport_number": ["A12345678"],
#         }
#         entities: list[PiiEntity] = []
#         for label, values in targets.items():
#             for value in values:
#                 start = page.page_text.find(value)
#                 if start >= 0:
#                     entities.append(PiiEntity(label, start, start + len(value), 0.95))
#         return tuple(entities)

# ocr_engine = SampleOcrEngine()
# pii_detector = SamplePiiDetector()
ocr_engine = PaddleOcrEngine()
pii_detector = Gliner2PiiDetector()
{"ocr": ocr_engine.model, "pii": pii_detector.model}

## 4. Upload the sample files and run analysis

Each sample gets a persistent queued job record, a source object in Floci S3, and a full `analyze_source` run. The returned jobs should be `ready` with one completed checkpoint per page.

In [ ]:
def source_type_for(path: Path) -> SourceType:
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return SourceType.PDF
    if suffix in {".jpg", ".jpeg"}:
        return SourceType.JPEG
    if suffix == ".png":
        return SourceType.PNG
    raise ValueError(f"unsupported source type: {path}")


def make_job(job_id: str, source_type: SourceType) -> Job:
    prefix = f"users/{OWNER_ID}/jobs/{job_id}/"
    return Job(
        id=job_id,
        owner_id=OWNER_ID,
        source_type=source_type,
        source_key=f"{prefix}source",
        output_key=f"{prefix}output",
        status=JobStatus.QUEUED,
        page_count=0,
        completed_pages=0,
        created_at=NOW,
        expires_at=NOW + timedelta(hours=24),
        model_versions={},
        failure_code=None,
        version=1,
    )

samples = {
    "sample-pdf": repo_root / "tests" / "samples" / "dlptest-name-dob-email.pdf",
    "passport-jpeg": repo_root / "tests" / "samples" / "Passport_of_Austria_(2024)_data_page.jpg",
}

ready_jobs: dict[str, Job] = {}
for job_id, sample_path in samples.items():
    clear_notebook_job(job_id)
    source = sample_path.read_bytes()
    job = make_job(job_id, source_type_for(sample_path))
    jobs.create(job)
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=job.source_key,
        Body=source,
        ContentType="application/octet-stream",
    )
    ready_jobs[job_id] = analyze_source(
        job,
        source,
        jobs=jobs,
        now=NOW,
        raster_writer=objects,
        ocr_store=objects,
        ocr_engine=ocr_engine,
        pii_detector=pii_detector,
        manifest_writer=objects,
    )

[
    {
        "job_id": job.id,
        "status": job.status.value,
        "page_count": job.page_count,
        "completed_pages": job.completed_pages,
        "model_versions": job.model_versions,
    }
    for job in ready_jobs.values()
]

## 5. Inspect persisted manifests

The worker wrote one page manifest per rasterized page. The table below summarizes the persisted suggestions without displaying OCR text or raw detected values from storage.

In [ ]:
manifest_rows = []
for job_id, job in ready_jobs.items():
    for page_number in range(1, job.page_count + 1):
        stored = objects.get_manifest(OWNER_ID, job.id, page_number)
        manifest = stored.manifest
        manifest_rows.append(
            {
                "job_id": job_id,
                "page": page_number,
                "version": manifest.version,
                "regions": len(manifest.suggestions),
                "categories": sorted({region.category for region in manifest.suggestions}),
            }
        )
manifest_rows

## 6. Draw and display redaction regions

The overlays are drawn on the rasterized PNG page artifacts stored in Floci S3. Coordinates come from the persisted `PageManifest` normalized region rectangles.

In [ ]:
COLORS = {
    "full_name": "#1f77b4",
    "date_of_birth": "#d62728",
    "email": "#2ca02c",
    "passport_number": "#9467bd",
}


def draw_manifest_overlay(job: Job, page_number: int) -> Image.Image:
    key = page_artifact_key(job.owner_id, job.id, page_number)
    page_bytes = s3.get_object(Bucket=BUCKET_NAME, Key=key)["Body"].read()
    image = Image.open(io.BytesIO(page_bytes)).convert("RGB")
    draw = ImageDraw.Draw(image)
    manifest = objects.get_manifest(job.owner_id, job.id, page_number).manifest

    for region in manifest.suggestions:
        color = COLORS.get(region.category, "#ff7f0e")
        left = region.x * image.width
        top = region.y * image.height
        right = (region.x + region.width) * image.width
        bottom = (region.y + region.height) * image.height
        draw.rectangle((left, top, right, bottom), outline=color, width=4)
        label_y = max(0, top - 16)
        draw.text((left, label_y), region.category, fill=color)
    return image

for job_id, job in ready_jobs.items():
    display(Markdown(f"### {job_id}"))
    for page_number in range(1, job.page_count + 1):
        display(Markdown(f"Page {page_number}"))
        display(draw_manifest_overlay(job, page_number))

## 7. Optional: rerun from a checkpoint

This cell simulates a retry after a completed first page by moving the persisted job back to `analyzing` with `completed_pages = 1`. On rerun, the worker loads the existing page-artifact index, skips page 1, processes only remaining pages, and returns to `ready`.

In [ ]:
pdf_job = ready_jobs["sample-pdf"]
if pdf_job.page_count > 1:
    checkpoint = pdf_job.model_copy(
        update={
            "status": JobStatus.ANALYZING,
            "completed_pages": 1,
            "version": pdf_job.version + 1,
        }
    )
    jobs.save(checkpoint, expected_version=pdf_job.version)
    resumed = analyze_source(
        checkpoint,
        samples["sample-pdf"].read_bytes(),
        jobs=jobs,
        now=NOW,
        raster_writer=objects,
        ocr_store=objects,
        ocr_engine=ocr_engine,
        pii_detector=pii_detector,
        manifest_writer=objects,
    )
    {
        "job_id": resumed.id,
        "status": resumed.status.value,
        "completed_pages": resumed.completed_pages,
        "page_count": resumed.page_count,
    }
else:
    "sample PDF has one page; checkpoint resume demo skipped"